In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import fastf1
import os
import requests
import tqdm


In [2]:
def get_fp1_best(FP1):

    laps = FP1.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds()

    return best

In [3]:
def get_fp2_best(FP2):

    laps = FP2.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds()

    return best

In [4]:
def get_fp3_best(FP3):

    laps = FP3.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds()

    return best

In [5]:
def get_fp2_longrun(FP2):

    laps = FP2.laps.pick_quicklaps()

    stints = laps.groupby(["Driver", "Stint"])

    longrun = (
        stints.filter(lambda x: len(x) >= 5)
        .groupby("Driver")["LapTime"]
        .mean()
        .dt.total_seconds()
    )

    return longrun

In [6]:
def get_sector_times (FP2):

    laps = FP2.laps

    sector_times = laps.groupby("Driver")[[
    "Sector1Time",
    "Sector2Time",
    "Sector3Time"
    ]].mean().apply(lambda x: x.dt.total_seconds())

    return sector_times

In [7]:
def get_average_laptime (FP2):

    laps = FP2.laps

    avg_lap = laps.groupby("Driver")["LapTime"].mean().dt.total_seconds()

    return avg_lap

In [8]:
def get_stint (FP2):

    laps = FP2.laps

    stints = laps.groupby(["Driver", "Stint"]).size().reset_index(name="LapCount")

    return stints

In [9]:
def get_tyre_compound (FP2):

    laps = FP2.laps

    tyres = laps.groupby("Driver")["Compound"].agg(lambda x: x.mode()[0])

    return tyres

In [10]:
def get_starting_grid_position (Result):

    grid = Result.results[["Abbreviation","GridPosition"]]

    return grid

In [11]:
def get_season_points(Result, race):

    data = []

    for r in range(1, race):

        race_points = Result.results[["Abbreviation","Points"]]

        data.append(race_points)

    df = pd.concat(data)

    return df.groupby("Abbreviation")["Points"].sum()

In [12]:
def get_season_avg_finish(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["Abbreviation", "Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    season_avg_finish = df.groupby("Abbreviation")["Position"].mean()

    return season_avg_finish

In [13]:
def get_constructor_position(Result, race):

    all_results = []

    for r in range(1, race):

        results = Result.results[["TeamName","Points"]]

        all_results.append(results)

    df = pd.concat(all_results)

    team_points = df.groupby("TeamName")["Points"].sum()

    constructor_position = (
        team_points
        .sort_values(ascending=False)
        .reset_index()
    )

    constructor_position["ConstructorPosition"] = (
        constructor_position.index + 1
    )

    return constructor_position[["TeamName","ConstructorPosition"]]

In [14]:
def get_driver_dnf_rate(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["Abbreviation","Status"]]

        results_list.append(results)

    df = pd.concat(results_list)

    df["DNF"] = df["Status"] != "Finished"

    dnf_rate = (
        df.groupby("Abbreviation")["DNF"]
        .mean()
        .reset_index()
        .rename(columns={"DNF": "DriverDNFRate"})
    )

    return dnf_rate

In [15]:
def get_team_dnf_rate(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["TeamName", "Status"]]

        results_list.append(results)

    df = pd.concat(results_list)

    df["DNF"] = df["Status"] != "Finished"

    team_dnf_rate = (
        df.groupby("TeamName")["DNF"]
        .mean()
        .reset_index()
        .rename(columns={"DNF": "TeamDNFRate"})
    )

    return team_dnf_rate

In [16]:
def get_team_avg_finish(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["TeamName","Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    team_avg_finish = (
        df.groupby("TeamName")["Position"]
        .mean()
        .reset_index()
        .rename(columns={"Position":"TeamAverageFinish"})
    )

    return team_avg_finish

In [17]:
def get_constructor_championship_position(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["TeamName","Points"]]

        results_list.append(results)

    df = pd.concat(results_list)

    team_points = df.groupby("TeamName")["Points"].sum().reset_index()

    team_points = team_points.sort_values("Points", ascending=False)

    team_points["ConstructorChampionshipPosition"] = range(1, len(team_points)+1)

    return team_points[["TeamName","ConstructorChampionshipPosition"]]

In [18]:
def get_rolling_team_performance(Result, race, window=3):

    results_list = []

    start_round = max(1, race - window)

    for r in range(start_round, race):

        results = Result.results[["TeamName","Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    rolling_performance = (
        df.groupby("TeamName")["Position"]
        .mean()
        .reset_index()
        .rename(columns={"Position":"RollingTeamPerformance"})
    )

    return rolling_performance

In [ ]:
def get_qualifying_data(Qualify):

    return Qualify.results[["Abbreviation","Q3"]]

In [20]:
def get_race_results(Result):

    return Result.results[["Abbreviation","Position"]]

In [ ]:
season = 2022

race = 5

FP1 = fastf1.get_session(season, race, 'FP1')
FP2 = fastf1.get_session(season, race, 'FP2')
FP3 = fastf1.get_session(season, race, 'FP3')

Result = fastf1.get_session(season, race, 'R')

Qualify = fastf1.get_session(season, race, 'Q')

FP1.load()
FP2.load()
FP3.load()

Result.load()

Qualify.load()

print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)

longrun = get_fp2_longrun(FP2)
sector_times = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre_compound = get_tyre_compound(FP2)
starting_grid_position = get_starting_grid_position(FP2)
season_points = get_season_points(FP2)
season_avg_finish = get_season_avg_finish(FP2)
constructor_position = get_constructor_position(Result, race)

driver_dnf_rate = get_driver_dnf_rate(Result, race)
team_dnf_rate = get_team_dnf_rate(Result, race)
team_avg_finish = get_team_avg_finish(Result, race)
constructor_championship_position = get_constructor_championship_position(Result, race)

rolling_team_performance = get_rolling_team_performance(Result, race, window=3)
qualifying_data = get_qualifying_data(Qualify)
race_results = get_race_results(Result)





req         WARNING 	DEFAULT CACHE ENABLED! (3.16 GB) /home/satyam/.cache/fastf1


In [ ]:
dataset = []

for season in range(2019, 2025):
    for race in range (1,23):
        try:
            results = get_race_results(season, race)

            qual = get_qualifying_data(season, race)

            merged = results.merge(qual, on="Abbreviation", how="left")

            merged["Season"] = season
            merged["Round"] = race

            dataset.append(merged)

            print(f"{season}, {race} Done.")
        except:
            continue

req         WARNING 	DEFAULT CACHE ENABLED! (1.9 GB) /home/satyam/.cache/fastf1
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 77 completed the race distance 00:00.387000 before the recorded end of the session.
core           INFO 	Finished loading data 

2019, 1 Done.


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 44 completed the race distance 00:00.070000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '16', '33', '5', '4', '7', '10', '23', '11', '99', '26', '20', '18', '63', '88', '27', '3', '55', '8']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data..

2019, 2 Done.


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 44 completed the race distance 00:00.423000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '5', '33', '16', '10', '3', '11', '7', '23', '8', '18', '20', '55', '99', '63', '88', '4', '26', '27']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data..

...

req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Driver 63: Ignoring late data f

2019, 21 Done.


core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

2020, 1 Done.


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Failed to align laps for driver

[]